# ViSceT5 + TWC — Finetune / Predict ở commit tuỳ chọn · Colab

**QA-ViT và Consformer theo đúng lý thuyết (code `f6b0581`, mỗi sửa một công tắc ở cell 2, mọi công tắc bắt đầu đúng bằng model khi tắt nó):**

| Công tắc | Vấn đề | Cách sửa |
|---|---|---|
| `QAVIT_GATED_JOINT` | Q1: softmax chung [ảnh, câu hỏi] đi qua `out_proj` không có cổng → ở β = 0 đặc trưng CLIP đã đổi ~7% | `CLIP(x) + tanh(β)·W′(λ(C_q − C_v))`, λ = phần attention mà câu hỏi lấy: β = 0 đúng CLIP (từng bit), tanh β = 1 đúng attention của paper |
| `QAVIT_GROUND` | Q2: cổng không mở vì adapter chỉ nhận gradient qua cổng (= 0) | loss −log(khối xác suất của patch dưới box OCR đáp án), trên logit λ (không qua cổng) → adapter học “nhìn đâu” trước khi cổng mở; trọng số `QAVIT_GROUND_WEIGHT` |
| `OCR_VISUAL_POOL` | token OCR không nhận đặc trưng CLIP (hỏi-theo-câu-hỏi) của vùng của nó | token += W·LN(trung bình patch dưới box), W = 0 lúc đầu |
| `CONSFORMER_SPAN` | C3: link chỉ học từ chữ, không giám sát | BCE trên link lớp cuối: trong span đáp án → 1, mép span → 0 (span = 1 constituent); trọng số `CONSFORMER_SPAN_WEIGHT` |
| `CONSFORMER_GEOMETRY` | C3: hai từ kề nhau trong chuỗi chỉ vì thứ tự đọc | bias của link từ (khoảng cách ngang, lệch dọc, tỉ lệ chiều cao, cùng từ) của hai box, lớp cuối MLP = 0 |
| `CONSFORMER_WORD_LINK` | các mảnh subword của một từ OCR có thể bị tách | link giữa chúng = 1 ở mọi lớp |

Log train có thêm `ground_loss`, `ground_mass` (so với `ground_area` = khối khi map đều), `span_loss`, `span_acc`, `qavit_gate` (trung bình |tanh β|), `ocr_visual_norm`.

**Các sửa trước (code `85ffb12`):**

| Công tắc | Lỗi được sửa | Cách sửa |
|---|---|---|
| `CONSFORMER_GATE` | C1: `LN(ConsFormer)` của 3 lớp ngẫu nhiên được cộng nguyên độ lớn vào embedding pretrained ngay từ bước 0 | `emb + g ⊙ LN(ConsFormer)`, `g` khởi tạo 0 (ReZero / LayerScale): bước 0 trùng từng trọng số với run không Consformer |
| `CONSFORMER_TREE` | C4: 12 cây mỗi lớp (chia head), không có vị trí | 1 cây mỗi lớp, chia `d_model`, vị trí sinusoid như Tree Transformer |
| `TWC_SIGLIP` | T1: BCE trung bình trên mọi ô, không bias → cặp đúng ~0.1% số ô, loss giảm bằng cách đẩy mọi cặp xuống, τ là lối tắt | `t·cos + b`, BCE cộng theo hàng, trung bình theo hàng (SigLIP) |
| `TWC_TEXT_ONLY` | T2: bản related dùng chung box/det/rec với từ gốc → nhận ra cặp nhờ vị trí, không cần đọc chữ | TWC trên đặc trưng chữ (token/ConsFormer + ký tự), anchor = từ bị đổi; embedding (dùng chung với lớp ra của decoder) không nhận gradient TWC |
| `TWC_TEXT_LABELS` | T3: cùng một chuỗi chữ ở ảnh khác bị gán là cặp âm | Cặp có cùng chuỗi chữ được bỏ qua |

Log train có thêm `twc_pos_logit`, `twc_neg_logit`, `twc_bias`, `consformer_gate` (trung bình |g|). Thứ tự chạy: (1) chỉ Consformer có cổng, so với base `cc773b0`; (2) chỉ TWC với ba sửa T1–T3; mỗi run so theo cặp trên test.

Notebook clone **repo GitHub `GIT_REPO`** và chuyển code sang đúng **`GIT_COMMIT`** (mã commit, tag hoặc tên nhánh), rồi train / predict bằng chính code đó: `setup.sh` → `run_pipeline.py` → `training/finetune.py configs/finetune.yaml` → push checkpoint mỗi epoch lên HF. Thông số không có trong cell 2 lấy từ `configs/finetune.yaml` **của commit đó**.

Mặc định: commit **`cc773b0`** (nhánh `exp/twc-bs4x2-ablation` = code `4221b1f` của run `Kus669/ViSceT5-twcbr-vitextvqa-consformer-twc-p35-tau0.2-bs4x2` + đặc trưng OCR đầy đủ cho mọi run) và cấu hình của run đó (`checkpoint-21975`: dev 30.32 / 56.68, test 31.64 / 56.09). Khi bật Consformer, model ở `cc773b0` được dựng giống hệt `4221b1f`; tham số train trùng `training_args.bin` của run đó.

**Thực nghiệm đối chứng (`TWC_CONTRASTIVE = False`):** code `0d1e4d0` = `1e5b8ef` + công tắc này, cùng cấu hình và seed với run Full `Kus669/ViSceT5-1e5b8ef-vitextvqa-consformer-twc-p35-tau0.07-bs32x1` (dev 32.20 / 57.61, test 33.27 / 56.79). Vẫn có bản OCR related, hai lượt encoder và loss trả lời trên cả hai. TWC bị bỏ hoàn toàn: không có tham số τ, không sinh nhãn, không tính `_twc`, không có `twc_loss` / `twc_acc` trong log.

Từ `1e5b8ef`, **TWC train trên hai lượt encoder riêng**: `[câu hỏi | ảnh | OCR gốc]` và `[câu hỏi | ảnh | OCR related]`, loss trả lời trên cả hai, TWC giữa hai lượt, nhiễu rút mới mỗi lần (`twc_two_views`). Lúc val / test / predict chỉ có OCR gốc, nên input giống hệt run chỉ bật Consformer. Copy head của `37a73f4` đã được gỡ; các run đó predict bằng code `0bbd575`.

Từ `b744ce2`, khi bật Consformer, **chữ OCR = embedding token ViT5 + đầu ra ConsFormer** (residual). ConsFormer bổ sung ngữ cảnh lên embedding pretrained thay vì thay thế nó. Ở `377099e`, đầu ra ConsFormer thay hẳn embedding.

Từ `377099e`, **ConsFormer tính đúng prior thành phần của Tree Transformer / ViConsFormer** (C[i, j] = tích các a[k, k+1] từ i đến j−1) và có output projection sau self-attention có cổng. Ở mọi commit cũ hơn, prior này ≈ 0 ngoài đường chéo nên self-attention của ConsFormer gần như không đóng góp (mỗi token OCR được mã hoá riêng lẻ): các run có `USE_CONSFORMER=True` trước `377099e` cần chạy lại. Run không bật Consformer không bị ảnh hưởng.

Từ `a04a02e`, khi bật TWC, **bản OCR related lúc val / test / predict được sửa 100%** (mọi từ sửa được đều sửa, không nhiễu ký tự). Lúc train vẫn là tăng cường TWA (giữ / nhiễu / sửa theo `TWC_AUG_P`) để học TWC. Ở commit cũ hơn, lúc test bản related vẫn theo quy trình lúc train (có nhiễu).

Từ `cc773b0`, **mọi run đều có đầy đủ đặc trưng OCR** (chữ + ký tự + box + det + rec):

| Cấu hình | Input của ViT5 encoder |
|---|---|
| baseline (`USE_CONSFORMER=False, USE_TWC=False`) | ảnh + câu hỏi + OCR gốc (embedding token ViT5 + ký tự + box + det + rec) |
| + Consformer | như baseline, nhưng chữ OCR qua ConsFormer thay cho embedding token |
| + TWC | như baseline + bản OCR related (chữ của bản related + ký tự + box/det/rec dùng chung với từ gốc) và loss TWC |
| + Consformer + TWC | cả hai (= run bs4x2) |

Ở commit cũ hơn (`4221b1f`), tắt Consformer nghĩa là OCR chỉ có chữ (không ký tự, box, det, rec).

| Biến (cell 2) | Ý nghĩa |
|---|---|
| `GIT_REPO`, `GIT_COMMIT` | repo GitHub và commit / tag / nhánh của code dùng để train và predict |
| `QAVIT` | `"off"`: CLIP thường, không adapter · `"no_question"`: adapter QA-ViT ở 6 lớp cuối CLIP được học, câu hỏi rỗng (chỉ có ở commit đọc `QAVIT_MODE`) · `"question"`: QA-ViT đầy đủ |
| `USE_CONSFORMER` | `True`: chữ OCR qua ConsFormer · `False`: embedding token ViT5 (từ `cc773b0` vẫn có ký tự + box + det + rec; ở commit cũ hơn là OCR text-only) |
| `USE_TWC` | `True`: thêm bản OCR related (OCR Augmentation) và loss TWC · `False`: chỉ OCR gốc |
| `TWC_AUG_P` | xác suất một từ của bản related bị đổi |
| `TWC_CONTRASTIVE` | `True` = Full · `False` = hai lượt + bản related, không có gì của TWC (`USE_TWC` vẫn `True` để có bản related; `TWC_TAU` không còn tác dụng) |
| `TWC_TAU` | temperature của TWC (học được hay cố định tuỳ code của commit) |
| `LEARNING_RATE` | learning rate phần pretrain (sửa từ `3e-5` thành `1e-4`; new modules giữ `1e-4`, scalar giữ `1e-3`) |
| `BATCH_SIZE`, `GRAD_ACCUM` | batch mỗi bước (`32`) và số bước tích luỹ gradient (`1`) (batch hiệu dụng = tích = 32) |
| `DATASET`, `PREDICT_CKPT` | dataset và checkpoint dùng ở mục 6 / 8 |

Cell 3 đọc `run_pipeline.py` của commit để biết nó nhận những biến môi trường nào, chỉ đặt các biến đó và **dừng** nếu một công tắc không áp dụng được ở commit này (ví dụ `QAVIT="no_question"` ở commit chưa có chế độ đó, hoặc batch khác yaml ở commit chưa đọc biến batch).
Tên run gồm mã commit: thư mục `output/<commit>_<DATASET>_<RUN_NAME>` và repo HF `ViSceT5-<commit>-<dataset>-<RUN_NAME>`, nên run ở commit khác nhau không bao giờ resume nhầm của nhau.

Ví dụ ablation chạy song song (mỗi run một phiên Colab, cùng `GIT_COMMIT = "cc773b0"`): `USE_CONSFORMER=False, USE_TWC=False` (baseline) và `USE_CONSFORMER=False, USE_TWC=True` (baseline + TWC).

## 1. Hugging Face token (để `finetune.py` tự push checkpoint mỗi epoch, tự resume khi Colab ngắt, và upload cuối)
Thêm secret `HF_TOKEN` (tab 🔑, bật *Notebook access*), **hoặc** chạy cell dưới rồi dán token.

In [ ]:
import os
try:
    from google.colab import userdata
    _t = userdata.get("HF_TOKEN")
    if _t:
        os.environ["HF_TOKEN"] = _t
except Exception as e:
    print("Colab Secrets:", e)
if not os.environ.get("HF_TOKEN"):
    from huggingface_hub import notebook_login
    notebook_login()
else:
    print("✅ Đã có HF_TOKEN từ Colab Secrets.")

## 2. Cấu hình — code (repo + commit) và module

In [ ]:
GIT_REPO       = "https://github.com/Kussssssss/ViSceT5.git"
GIT_COMMIT     = "f6b0581"
QAVIT          = "question"

# =========================================================================
# CHỌN 1 TRONG 4 CẤU HÌNH ABLATION (từ commit cc773b0: cả 4 cấu hình đều có đầy đủ Image + Question + OCR text + bbox + det + rec + char):
# 1. Baseline:                    USE_CONSFORMER = False, USE_TWC = False
# 2. Baseline + Consformer:       USE_CONSFORMER = True,  USE_TWC = False
# 3. Baseline + TWC:              USE_CONSFORMER = False, USE_TWC = True
# 4. Baseline + Consformer + TWC: USE_CONSFORMER = True,  USE_TWC = True (Full model)
# =========================================================================
USE_CONSFORMER = True
USE_TWC        = True     # True + TWC_CONTRASTIVE = False = DOVE (hai lượt gốc / related, không TWC)
TWC_AUG_P      = 0.35
TWC_TAU        = 0.1
TWC_CONTRASTIVE = False   # True = có TWC · False = hai lượt + bản related, bỏ hoàn toàn TWC (DOVE)
TWC_TWO_VIEWS   = True    # khi bật TWC: True = hai lượt encoder [gốc] và [related] · False = một lượt [gốc | related] như TWA
# Các sửa lỗi (code 85ffb12), mỗi cái một công tắc:
CONSFORMER_GATE = True    # C1: ConsFormer cộng vào embedding qua cổng g = 0 lúc đầu → bước 0 đúng bằng run không Consformer
CONSFORMER_TREE = False   # C4: group attention như Tree Transformer (1 cây mỗi lớp) + vị trí sinusoid; bật sau khi cổng đạt base
TWC_SIGLIP      = True    # T1: logit = t·cos + b (b0 = −1/τ0), BCE cộng theo hàng, trung bình theo hàng; dùng TWC_TAU = 0.1 (t0 = 10, b0 = −10 như SigLIP)
TWC_TEXT_ONLY   = True    # T2: TWC trên đặc trưng chữ (không box/det/rec), anchor = từ bị đổi; cần TWC_TEXT_LABELS
TWC_TEXT_LABELS = True    # T3: cặp có cùng chuỗi chữ không làm cặp âm (bỏ qua)
# QA-ViT và Consformer (code f6b0581); chỉ có tác dụng khi module tương ứng bật:
QAVIT_GATED_JOINT = True  # Q1: β = 0 đúng CLIP; tanh β = 1 đúng attention chung của paper
QAVIT_GROUND      = True  # Q2: loss grounding trên box OCR của đáp án (cần QAVIT_GATED_JOINT)
QAVIT_GROUND_WEIGHT = 0.1
OCR_VISUAL_POOL   = False # token OCR += W·LN(patch CLIP dưới box), W = 0 lúc đầu (cần ảnh)
CONSFORMER_SPAN   = True  # C3: span đáp án = một constituent (loss trên link lớp cuối)
CONSFORMER_SPAN_WEIGHT = 0.1
CONSFORMER_GEOMETRY  = True  # C3: link thấy bố cục box (bias khởi tạo 0)
CONSFORMER_WORD_LINK = True  # subword của cùng một từ OCR luôn nối
LEARNING_RATE  = 1e-4   # phần pretrain sửa từ 3e-5 thành 1e-4 (new modules giữ 1e-4, scalar giữ 1e-3)
BATCH_SIZE     = 32
GRAD_ACCUM     = 1
DATASET        = "ViTextVQA"
PREDICT_CKPT   = "best"
PREDICT_MIN_DEV_EM = 31.0  # run đối chứng: 0.0 (điểm thấp cũng là kết quả cần có); run cải tiến: 31.0

import os
from huggingface_hub import get_token, HfApi
_tok = os.environ.get("HF_TOKEN") or get_token()
if _tok:
    os.environ["HF_TOKEN"] = _tok
HF_USER = "Kus669"
if _tok:
    try:
        HF_USER = HfApi(token=_tok).whoami().get("name") or HF_USER
    except Exception as e:
        print("whoami lỗi:", e)
else:
    print("⚠️ CHƯA có HF_TOKEN: checkpoint chỉ nằm trên Colab, mất khi ngắt kết nối!")

assert QAVIT in ("off", "no_question", "question"), "QAVIT: off | no_question | question"
assert 0 <= TWC_AUG_P <= 1 and TWC_TAU > 0 and BATCH_SIZE >= 1 and GRAD_ACCUM >= 1 and LEARNING_RATE > 0
assert not (USE_TWC and TWC_CONTRASTIVE and TWC_TEXT_ONLY and not TWC_TEXT_LABELS), "TWC_TEXT_ONLY cần TWC_TEXT_LABELS = True"
assert not (QAVIT == "question" and QAVIT_GROUND and not QAVIT_GATED_JOINT), "QAVIT_GROUND cần QAVIT_GATED_JOINT = True"
assert QAVIT_GROUND_WEIGHT > 0 and CONSFORMER_SPAN_WEIGHT > 0
_qv = QAVIT == "question"
_tag = lambda name, on, weight=None: (f"-{name}" + (f"{weight:g}" if weight not in (None, 0.1) else "")) if on else ""
if USE_TWC and TWC_CONTRASTIVE and TWC_SIGLIP and TWC_TAU != 0.1:
    print(f"⚠️ TWC_SIGLIP với TWC_TAU = {TWC_TAU}: t0 = {1 / TWC_TAU:g}, b0 = {-1 / TWC_TAU:g} (SigLIP dùng 0.1 → t0 = 10, b0 = −10)")
RUN_NAME = "-".join([m for m, on in (("qavit" + _tag("gj", QAVIT_GATED_JOINT) + _tag("gr", QAVIT_GATED_JOINT and QAVIT_GROUND, QAVIT_GROUND_WEIGHT), _qv),
                                     ("qavitnq", QAVIT == "no_question"),
                                     ("consformer" + ("-gate" if CONSFORMER_GATE else "") + ("-tt" if CONSFORMER_TREE else "")
                                      + _tag("span", CONSFORMER_SPAN, CONSFORMER_SPAN_WEIGHT) + _tag("geo", CONSFORMER_GEOMETRY)
                                      + _tag("wl", CONSFORMER_WORD_LINK), USE_CONSFORMER),
                                     ((f"twc-p{int(round(100 * TWC_AUG_P))}-tau{TWC_TAU:g}" + ("-siglip" if TWC_SIGLIP else "")
                                       + ("-text" if TWC_TEXT_ONLY else "") + ("-tlab" if TWC_TEXT_LABELS else "")
                                       if TWC_CONTRASTIVE else f"twoviews-p{int(round(100 * TWC_AUG_P))}")
                                      + ("" if TWC_TWO_VIEWS else "-onepass"), USE_TWC),
                                     ("vpool", OCR_VISUAL_POOL)) if on] or ["base"])
RUN_NAME += f"-bs{BATCH_SIZE}x{GRAD_ACCUM}"
print(f"Code: {GIT_REPO} @ {GIT_COMMIT}")
print(f"Cấu hình: QA-ViT={QAVIT} | Consformer={USE_CONSFORMER} | TWC={USE_TWC}"
      f"{f' (p={TWC_AUG_P}, τ={TWC_TAU}, TWC_CONTRASTIVE={TWC_CONTRASTIVE}, hai lượt={TWC_TWO_VIEWS}, SigLIP={TWC_SIGLIP}, chữ={TWC_TEXT_ONLY}, nhãn chữ={TWC_TEXT_LABELS})' if USE_TWC else ''}"
      f"{f' | cổng ConsFormer={CONSFORMER_GATE}, Tree Transformer={CONSFORMER_TREE}, span={CONSFORMER_SPAN}, hình học={CONSFORMER_GEOMETRY}, nối subword={CONSFORMER_WORD_LINK}' if USE_CONSFORMER else ''}"
      f"{f' | QA-ViT gated joint={QAVIT_GATED_JOINT}, grounding={QAVIT_GROUND}' if _qv else ''} | box-pooled visual={OCR_VISUAL_POOL} | lr={LEARNING_RATE:g} | batch={BATCH_SIZE}x{GRAD_ACCUM} | RUN_NAME={RUN_NAME}")

## 3. Lấy code đúng commit và đặt biến môi trường
Clone `GIT_REPO` (lần đầu), chuyển sang `GIT_COMMIT`, chạy `setup.sh` của commit đó, rồi đặt biến môi trường theo những gì `run_pipeline.py` của commit đó đọc được. Chạy lại cell này mỗi khi đổi cell 2.

In [ ]:
import os, re, subprocess, yaml
REPO_DIR = "/content/ViSceT5"
if not os.path.exists(os.path.join(REPO_DIR, ".git")):
    !git clone -q {GIT_REPO} {REPO_DIR}
%cd {REPO_DIR}
!git remote set-url origin {GIT_REPO}
!git fetch -q origin "+refs/heads/*:refs/remotes/origin/*" --tags


def _resolve(ref):
    for cand in (f"origin/{ref}", ref):
        out = subprocess.run(["git", "rev-parse", "--verify", "-q", cand + "^{commit}"], capture_output=True, text=True)
        if out.returncode == 0:
            return out.stdout.strip()
    raise ValueError(f"Không tìm thấy commit/nhánh/tag '{ref}' trong {GIT_REPO}")


CODE_SHA = _resolve(GIT_COMMIT)
CODE_TAG = CODE_SHA[:7]
subprocess.run(["git", "checkout", "-q", "--detach", CODE_SHA], check=True)
!git log --oneline -1
!bash setup.sh

READ = set(re.findall(r'"([A-Z][A-Z_]+)": "[a-z_]+"', open("run_pipeline.py", encoding="utf-8").read()))
YAML_CFG = yaml.safe_load(open("configs/finetune.yaml", encoding="utf-8"))
for _name in ("ABLATION_USE_IMAGE", "ABLATION_USE_OCR", "ABLATION_USE_OCR_AUG", "ABLATION_USE_QACLIP", "CLIP_LATE_ADAPTER",
              "DATASET_NAME", "GRADIENT_ACCUMULATION_STEPS", "HF_REPO", "LEARNING_RATE", "MOCK_TEST", "MODEL_NAME_OR_PATH",
              "NEW_MODULE_LR", "NUM_TRAIN_EPOCHS", "OCR_AUG_MODE", "OCR_FEATURE_NORM", "OCR_RELATED", "OCR_LATTICE_KEEP",
              "OUTPUT_DIR", "PER_DEVICE_EVAL_BATCH_SIZE", "PER_DEVICE_TRAIN_BATCH_SIZE", "QAVIT_MODE",
              "RESUME_CHECKPOINT_ID", "RESUME_FROM_CHECKPOINT", "SCALAR_LR", "STAGE", "TRAIN_CONFIG",
              "TWC_ADV_PROBABILITY", "TWC_ANSWER_WEIGHT", "TWC_CONTRASTIVE", "TWC_TWO_VIEWS", "TWC_SIGLIP", "TWC_TEXT_ONLY", "TWC_TEXT_LABELS", "CONSFORMER_GATE", "CONSFORMER_TREE_TRANSFORMER",
              "QAVIT_GATED_JOINT", "QAVIT_GROUND", "QAVIT_GROUND_WEIGHT", "CONSFORMER_SPAN", "CONSFORMER_SPAN_WEIGHT",
              "CONSFORMER_GEOMETRY", "CONSFORMER_WORD_LINK", "OCR_VISUAL_POOL", "TWC_FEATURE", "TWC_LOSS", "TWC_MASK_SAME_TEXT", "TWC_OBJECTIVE",
              "TWC_QUEUE_SIZE", "TWC_QUEUE_START", "TWC_RELATED_TRAIN", "TWC_TEMPERATURE", "TWC_VIEW", "TWC_WEIGHT",
              "USE_TWC", "GIT_BRANCH", "GIT_COMMIT", "PREDICT_CKPT_DIR", "PREDICT_DATASET", "PREDICT_SPLIT"):
    os.environ.pop(_name, None)

_b = lambda v: "true" if v else "false"
OUTPUT_DIR = f"/content/ViSceT5/output/{CODE_TAG}_{DATASET}_{RUN_NAME}"
HF_REPO = f"{HF_USER}/ViSceT5-{CODE_TAG}-{DATASET.lower()}-{RUN_NAME}"
ENV = {"DATASET_NAME": DATASET, "ABLATION_USE_OCR": _b(USE_CONSFORMER), "USE_TWC": _b(USE_TWC),
       "TWC_ADV_PROBABILITY": str(TWC_AUG_P), "TWC_TEMPERATURE": str(TWC_TAU)}
if USE_TWC and not TWC_CONTRASTIVE:
    ENV["TWC_CONTRASTIVE"] = "false"
if USE_TWC and not TWC_TWO_VIEWS:
    ENV["TWC_TWO_VIEWS"] = "false"
for _name, _on in (("CONSFORMER_GATE", USE_CONSFORMER and CONSFORMER_GATE),
                   ("CONSFORMER_TREE_TRANSFORMER", USE_CONSFORMER and CONSFORMER_TREE),
                   ("TWC_SIGLIP", USE_TWC and TWC_CONTRASTIVE and TWC_SIGLIP),
                   ("TWC_TEXT_ONLY", USE_TWC and TWC_CONTRASTIVE and TWC_TEXT_ONLY),
                   ("TWC_TEXT_LABELS", USE_TWC and TWC_CONTRASTIVE and TWC_TEXT_LABELS),
                   ("QAVIT_GATED_JOINT", QAVIT == "question" and QAVIT_GATED_JOINT),
                   ("QAVIT_GROUND", QAVIT == "question" and QAVIT_GATED_JOINT and QAVIT_GROUND),
                   ("CONSFORMER_SPAN", USE_CONSFORMER and CONSFORMER_SPAN),
                   ("CONSFORMER_GEOMETRY", USE_CONSFORMER and CONSFORMER_GEOMETRY),
                   ("CONSFORMER_WORD_LINK", USE_CONSFORMER and CONSFORMER_WORD_LINK),
                   ("OCR_VISUAL_POOL", OCR_VISUAL_POOL)):
    if _on:
        ENV[_name] = "true"
if ENV.get("QAVIT_GROUND"):
    ENV["QAVIT_GROUND_WEIGHT"] = str(QAVIT_GROUND_WEIGHT)
if ENV.get("CONSFORMER_SPAN"):
    ENV["CONSFORMER_SPAN_WEIGHT"] = str(CONSFORMER_SPAN_WEIGHT)
_missing = [k for k in ENV if k not in READ]
assert not _missing, f"Commit {CODE_TAG}: run_pipeline.py không đọc {_missing} → không áp dụng được cấu hình ở cell 2."
if "QAVIT_MODE" in READ:
    ENV["QAVIT_MODE"] = QAVIT
else:
    assert QAVIT != "no_question", f"Commit {CODE_TAG} chưa có QA-ViT no_question: đặt QAVIT = \"off\" hoặc \"question\"."
if "ABLATION_USE_QACLIP" in READ:
    ENV["ABLATION_USE_QACLIP"] = _b(QAVIT == "question")
if "ABLATION_USE_OCR_AUG" in READ:
    ENV["ABLATION_USE_OCR_AUG"] = _b(USE_TWC)
if "LEARNING_RATE" in READ and "LEARNING_RATE" in globals() and LEARNING_RATE is not None:
    ENV["LEARNING_RATE"] = str(LEARNING_RATE)
for _name, _key, _value in (("PER_DEVICE_TRAIN_BATCH_SIZE", "per_device_train_batch_size", BATCH_SIZE),
                            ("GRADIENT_ACCUMULATION_STEPS", "gradient_accumulation_steps", GRAD_ACCUM)):
    if _name in READ:
        ENV[_name] = str(_value)
    else:
        assert YAML_CFG.get(_key) == _value, (f"Commit {CODE_TAG} không đọc {_name}: batch lấy từ yaml "
                                              f"({_key} = {YAML_CFG.get(_key)}) → đặt BATCH_SIZE / GRAD_ACCUM đúng giá trị đó.")
os.environ.update(ENV, OUTPUT_DIR=OUTPUT_DIR, HF_REPO=HF_REPO, PYTHONUNBUFFERED="1")
print(f"Code {CODE_TAG} | biến môi trường: {ENV}")
print(f"OUTPUT_DIR={OUTPUT_DIR}\nHF_REPO={HF_REPO}\nPREDICT_CKPT={PREDICT_CKPT}")

## 4. Kiểm tra trước khi train
Cell đầu chạy unit test và dừng nếu có test lỗi. Cell sau là mock test `--smoke_test`: 8 mẫu train, 4 mẫu val, 3 bước; không push, không resume từ HF.


In [ ]:
!git checkout -q --detach {CODE_SHA}
!pip install -q pytest
_tests = subprocess.run(["python", "-m", "pytest", "tests/test_vqa_pipeline.py", "tests/test_qaclip.py", "-q", "--tb=short"],
                        capture_output=True, text=True)
print(_tests.stdout[-6000:], _tests.stderr[-1000:])
assert _tests.returncode == 0, "Có unit test lỗi: gửi output ở trên trước khi train."


In [ ]:
!git checkout -q --detach {CODE_SHA}
!HF_REPO= OUTPUT_DIR={OUTPUT_DIR}_mock MOCK_TEST=true STAGE=finetune python3 run_pipeline.py 2>&1 | tee train_execution.log

## 5. Finetune (từ đầu, không pretrain) — log train có `twc_loss`, `twc_acc`, `twc_tau`, `view_loss` khi bật TWC; `ground_loss`, `ground_mass`, `ground_area`, `qavit_gate` khi bật grounding; `span_loss`, `span_acc` khi bật span

In [ ]:
!git checkout -q --detach {CODE_SHA}
!MOCK_TEST=false STAGE=finetune python3 run_pipeline.py 2>&1 | tee train_execution.log

## 5b. (Khi cần) Đẩy checkpoint local lên HF rồi train tiếp
Dùng khi đã train mà **chưa có `HF_TOKEN`** (checkpoint chỉ nằm trên Colab): chạy cell 1 (token) → cell 2 (cấu hình, đúng commit và công tắc của lần train) → cell 3 → cell này. Cell đẩy các `checkpoint-N` trong `OUTPUT_DIR` (kèm optimizer/scheduler để resume) lên `HF_REPO`, sau đó chạy lại **cell 5**: `finetune.py` tự resume từ checkpoint mới nhất trên HF và từ đó push mỗi epoch. Chỉ checkpoint đã lưu cuối epoch mới có; phần đang train dở của epoch sau bị mất. Nếu runtime Colab đã reset thì không còn gì để đẩy.

In [ ]:
!ls -d {OUTPUT_DIR}/checkpoint-* 2>/dev/null
!python scripts/hf_io.py upload_checkpoints --repo {HF_REPO} --dir {OUTPUT_DIR}

## 6. Dự đoán dev + test (chỉ khi dev EM ở epoch 4 hoặc 5 đạt `PREDICT_MIN_DEV_EM` ở cell 2) (`STAGE=predict` của `run_pipeline.py`) và chấm điểm
`predict.py` in EM/F1 trên dev và ghi file nộp cho test. File được chép vào `OUTPUT_DIR` để không đè giữa các cấu hình.
Test được chấm bằng `training/benchmark_eval.py` — **đúng giao thức benchmark** (cùng cách chấm các dòng baseline/LMM trong bài): EM, F1 token/char, BLEU-1..4/avg, CIDEr, BERTScore (mBERT).
Đáp án test lấy từ `eval_gt/<DATASET>_test_gt.json` trong repo (đã có `ViTextVQA`); dataset chưa có file thì cell hiện nút upload **ngay đầu cell** (trước khi predict).
Nếu `OUTPUT_DIR` (hoặc `predictions/` trên HF) đã có `submission_test.csv` thì cell **bỏ qua predict** và chỉ chấm điểm.
File nộp, log predict (có EM/F1 dev) và kết quả chấm test được upload vào thư mục `predictions/` của repo HF của cấu hình.

**Chọn checkpoint:** `PREDICT_CKPT` ở cell 2. Cell in bảng các checkpoint kèm EM/F1/CIDEr trên val của từng epoch rồi dùng checkpoint được chọn:
`best` = model tốt nhất (thư mục gốc, do `trainer.save_model` lưu sau `load_best_model_at_end`), `latest`, `epoch3`, hoặc `checkpoint-17580`.
Chỉ tải file inference của checkpoint đó (không tải optimizer). Kết quả của `best` nằm ở `OUTPUT_DIR` và `predictions/` trên HF;
của checkpoint khác nằm ở `OUTPUT_DIR/predict_checkpoint-N` và `predictions/checkpoint-N/` → không đè nhau.

**Predict lại sau khi Colab đã tắt:** chạy cell 1, 2, 3 (đặt đúng commit, công tắc của lần train và `PREDICT_CKPT`) rồi chạy thẳng cell này, bỏ qua mock và finetune.

In [ ]:
import os, shutil, glob, json
from huggingface_hub import HfApi, hf_hub_download
!git checkout -q --detach {CODE_SHA}
_states = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint-*/trainer_state.json"), key=lambda p: int(p.split("checkpoint-")[-1].split("/")[0]))
if not _states:
    _remote = sorted((f for f in HfApi().list_repo_files(HF_REPO) if f.endswith("/trainer_state.json")), key=lambda p: int(p.split("checkpoint-")[-1].split("/")[0]))
    _states = [hf_hub_download(HF_REPO, _remote[-1])] if _remote else []
_devs = [100 * h["eval_em"] for h in json.load(open(_states[-1]))["log_history"] if "eval_em" in h] if _states else []
print("Dev EM theo epoch:", [round(d, 2) for d in _devs])
if max(_devs[3:5], default=0) < PREDICT_MIN_DEV_EM:
    raise SystemExit(f"Dev EM ở epoch 4, 5 chưa đạt {PREDICT_MIN_DEV_EM}: bỏ qua predict.")
TEST_GT_JSON = f"/content/ViSceT5/eval_gt/{DATASET}_test_gt.json"
if not os.path.exists(TEST_GT_JSON):
    TEST_GT_JSON = f"/content/{DATASET}_test_gt.json"
if not os.path.exists(TEST_GT_JSON):
    from google.colab import files
    print(f"⬆️ Chọn file {os.path.basename(TEST_GT_JSON)} để upload (bấm Cancel nếu chưa có → vẫn predict, chưa chấm test)")
    up = files.upload()
    if up:
        shutil.move(os.path.abspath(next(iter(up))), TEST_GT_JSON)
print("GT test:", TEST_GT_JSON if os.path.exists(TEST_GT_JSON) else "chưa có")

import json
if os.path.exists("/content/ckpt_info.json"):
    os.remove("/content/ckpt_info.json")
!python scripts/hf_io.py prepare --repo {HF_REPO} --dir {OUTPUT_DIR} --ckpt {PREDICT_CKPT} --info_out /content/ckpt_info.json
CKPT_INFO = json.load(open("/content/ckpt_info.json", encoding="utf-8"))
MODEL_DIR, PRED_DIR, PRED_REPO_PATH = CKPT_INFO["model_dir"], CKPT_INFO["pred_dir"], CKPT_INFO["pred_path_in_repo"]
assert os.path.exists(os.path.join(MODEL_DIR, "model.safetensors")), f"Không có model trong {MODEL_DIR}"

tag = "" if DATASET == "ViTextVQA" else f"{DATASET}_"
pred = os.path.join(PRED_DIR, f"submission_{tag}test.csv")
if os.path.exists(pred):
    print(f"✅ Đã có {pred} → bỏ qua predict, chỉ chấm điểm.")
else:
    for sp in ("dev", "test"):
        if os.path.exists(f"/content/ViSceT5/output/submission_{tag}{sp}.csv"):
            os.remove(f"/content/ViSceT5/output/submission_{tag}{sp}.csv")
    !MOCK_TEST=false STAGE=predict PREDICT_SPLIT=both PREDICT_DATASET={DATASET} PREDICT_CKPT_DIR={MODEL_DIR} HF_REPO= python3 run_pipeline.py 2>&1 | tee train_execution.log
    for sp in ("dev", "test"):
        src = f"/content/ViSceT5/output/submission_{tag}{sp}.csv"
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(PRED_DIR, os.path.basename(src)))
            print("→", os.path.join(PRED_DIR, os.path.basename(src)))
    shutil.copy2("/content/ViSceT5/train_execution.log", os.path.join(PRED_DIR, "predict_log.txt"))
log = os.path.join(PRED_DIR, "predict_log.txt")
dev_lines = [l.strip() for l in open(log, encoding="utf-8", errors="ignore") if "[dev] EM=" in l] if os.path.exists(log) else []
if dev_lines:
    for line in dev_lines[-2:]:
        print(line)
if os.path.exists(TEST_GT_JSON) and os.path.exists(pred):
    !pip install -q evaluate bert-score git+https://github.com/salaniz/pycocoevalcap
    !python -m training.benchmark_eval --gt {TEST_GT_JSON} --pred {pred} --out {PRED_DIR}/eval_test.json
else:
    print("ℹ️ Chưa đặt TEST_GT_JSON → chưa chấm test.")

if os.environ.get("HF_TOKEN") and HF_REPO:
    !python scripts/hf_io.py upload_predictions --repo {HF_REPO} --dir {PRED_DIR} --path_in_repo {PRED_REPO_PATH} --files submission_{tag}dev.csv submission_{tag}test.csv predict_log.txt eval_test.json
else:
    print("⚠️ Không có HF_TOKEN/HF_REPO → file dự đoán chỉ nằm trên Colab, hãy tải về trước khi ngắt phiên.")

## 6b. Dự đoán + chấm điểm một repo HF bất kỳ
Chọn thẳng repo model trên Hugging Face (`HF_PRED_REPO`) và checkpoint (`HF_PRED_CKPT`: `best` | `latest` | `epochN` | `checkpoint-N`), không phụ thuộc cấu hình ở cell 2. Các bước giống mục 6; kết quả upload vào `predictions/` (hoặc `predictions/checkpoint-N/`) của chính repo đó. Repo đã có file dự đoán của checkpoint đó thì chỉ chấm lại.

**`HF_PRED_COMMIT`:** commit code dùng để predict — phải là code đã train run đó, để model dựng được và bản OCR related được tạo đúng như lúc train. `"auto"` (mặc định):

| repo / config của checkpoint | commit |
|---|---|
| tên repo có mã commit (`ViSceT5-<commit>-…`, run của notebook này) | commit đó |
| checkpoint có `consformer_link_weight` hoặc `twc_copy` (run từ `37a73f4`) | `37a73f4` |
| checkpoint có `consformer_residual: true` (run từ `b744ce2`) | `b744ce2` |
| checkpoint có `consformer_constituent: tree` (run từ `377099e`) | `377099e` |
| checkpoint có `twc_eval_related: correct` (run từ `a04a02e`) | `a04a02e` |
| checkpoint có `ocr_full_features: true` (các run ablation từ `cc773b0`) | `cc773b0` |
| `ablation_use_ocr_aug=true`, `ocr_aug_mode="separate"` (nhánh `tmp`, ví dụ `Kus669/tmp_viscet5`) | `2090b2d` |
| `ablation_use_ocr_aug=true`, lattice / concat (ví dụ lat-p70) | `bf74388` |
| TWC bật, chưa có trường `clip_late_adapter` (run `…-consformer-twc-…`: bs8x2, bs4x2) | `4221b1f` (có trường box/layout: `5545dce` / `1046e86`) |
| còn lại | code của cell 3 |

Sau khi predict, code quay lại commit của cell 3.

In [ ]:
HF_PRED_REPO    = HF_REPO
HF_PRED_CKPT    = "best"
HF_PRED_COMMIT  = "auto"
HF_PRED_DATASET = DATASET

import os, re, shutil, json, subprocess
!git checkout -q --detach {CODE_SHA}
HF_PRED_DIR = f"/content/ViSceT5/output/hf_{HF_PRED_REPO.split('/')[-1]}"
TEST_GT_JSON = f"/content/ViSceT5/eval_gt/{HF_PRED_DATASET}_test_gt.json"
if not os.path.exists(TEST_GT_JSON):
    TEST_GT_JSON = f"/content/{HF_PRED_DATASET}_test_gt.json"
print("GT test:", TEST_GT_JSON if os.path.exists(TEST_GT_JSON) else "chưa có (chạy mục 6 một lần để upload)")

if os.path.exists("/content/hf_ckpt_info.json"):
    os.remove("/content/hf_ckpt_info.json")
!python scripts/hf_io.py prepare --repo {HF_PRED_REPO} --dir {HF_PRED_DIR} --ckpt {HF_PRED_CKPT} --info_out /content/hf_ckpt_info.json
HF_INFO = json.load(open("/content/hf_ckpt_info.json", encoding="utf-8"))
H_MODEL_DIR, H_PRED_DIR, H_REPO_PATH = HF_INFO["model_dir"], HF_INFO["pred_dir"], HF_INFO["pred_path_in_repo"]

h_tag = "" if HF_PRED_DATASET == "ViTextVQA" else f"{HF_PRED_DATASET}_"
h_pred = os.path.join(H_PRED_DIR, f"submission_{h_tag}test.csv")
h_cfg = json.load(open(os.path.join(H_MODEL_DIR, "config.json"), encoding="utf-8"))
h_commit = HF_PRED_COMMIT
h_named = re.match(r"ViSceT5-([0-9a-f]{7})-", HF_PRED_REPO.split("/")[-1])
if h_commit == "auto":
    if h_named:
        h_commit = h_named.group(1)
    elif h_cfg.get("twc_copy") or h_cfg.get("consformer_link_weight"):
        h_commit = "37a73f4"
    elif h_cfg.get("consformer_residual"):
        h_commit = "b744ce2"
    elif h_cfg.get("consformer_constituent") == "tree":
        h_commit = "377099e"
    elif h_cfg.get("twc_eval_related") == "correct":
        h_commit = "a04a02e"
    elif h_cfg.get("ocr_full_features"):
        h_commit = "cc773b0"
    elif h_cfg.get("ablation_use_ocr_aug"):
        h_commit = "2090b2d" if h_cfg.get("ocr_aug_mode") == "separate" else "bf74388"
    elif h_cfg.get("use_twc") and h_cfg.get("twc_objective") in ("answer", "both"):
        h_commit = "bf74388"
    elif h_cfg.get("use_twc") and "clip_late_adapter" not in h_cfg:
        h_commit = ("1046e86" if "ocr_related_remove_layout" in h_cfg
                    else "5545dce" if "ocr_related_remove_bbox" in h_cfg else "4221b1f")
    else:
        h_commit = ""
h_sha = _resolve(h_commit) if h_commit else CODE_SHA
print(f"Code dùng để predict: {h_sha[:7]}")
if os.path.exists(h_pred):
    print(f"✅ Đã có {h_pred} → bỏ qua predict, chỉ chấm điểm.")
else:
    for sp in ("dev", "test"):
        if os.path.exists(f"/content/ViSceT5/output/submission_{h_tag}{sp}.csv"):
            os.remove(f"/content/ViSceT5/output/submission_{h_tag}{sp}.csv")
    !git checkout -q --detach {h_sha}
    !MOCK_TEST=false STAGE=predict PREDICT_SPLIT=both PREDICT_DATASET={HF_PRED_DATASET} PREDICT_CKPT_DIR={H_MODEL_DIR} HF_REPO= python3 run_pipeline.py 2>&1 | tee train_execution.log
    !git checkout -q --detach {CODE_SHA}
    for sp in ("dev", "test"):
        src = f"/content/ViSceT5/output/submission_{h_tag}{sp}.csv"
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(H_PRED_DIR, os.path.basename(src)))
            print("→", os.path.join(H_PRED_DIR, os.path.basename(src)))
    shutil.copy2("/content/ViSceT5/train_execution.log", os.path.join(H_PRED_DIR, "predict_log.txt"))
h_log = os.path.join(H_PRED_DIR, "predict_log.txt")
h_dev = [l.strip() for l in open(h_log, encoding="utf-8", errors="ignore") if "[dev] EM=" in l] if os.path.exists(h_log) else []
for line in h_dev[-2:]:
    print(line)
if os.path.exists(TEST_GT_JSON) and os.path.exists(h_pred):
    !pip install -q evaluate bert-score git+https://github.com/salaniz/pycocoevalcap
    !python -m training.benchmark_eval --gt {TEST_GT_JSON} --pred {h_pred} --out {H_PRED_DIR}/eval_test.json
else:
    print("ℹ️ Chưa có GT test hoặc file dự đoán → chưa chấm test.")

if os.environ.get("HF_TOKEN") and os.path.exists(h_pred):
    !python scripts/hf_io.py upload_predictions --repo {HF_PRED_REPO} --dir {H_PRED_DIR} --path_in_repo {H_REPO_PATH} --files submission_{h_tag}dev.csv submission_{h_tag}test.csv predict_log.txt eval_test.json

## 6c. (Tuỳ chọn) So sánh 3 cách đưa OCR related lúc predict, cho checkpoint một lượt encoder cũ
Với mỗi checkpoint TWC trong `HF_CMP_RUNS` (mặc định: run có copy `0bbd575` và run không có copy `b744ce2`), predict dev và test với:

| `mode` | Bản related trong input của encoder |
|---|---|
| `none` | không có |
| `augment` | như lúc train: nhiễu / sửa với xác suất `twc_adv_probability` của checkpoint |
| `correct` | sửa 100% từ sửa được, không nhiễu |

Kết quả upload vào `predictions/related_<mode>/` của repo. Cell in EM, F1, độ dài câu trả lời, phân loại lỗi và chênh lệch có cặp giữa mọi cặp, gồm cả baseline `cc773b0`. Cell dùng code của cell 3 và không train. Chỉ cần chạy mục 1, 2, 3 rồi đến cell này.


In [ ]:
HF_CMP_RUNS = {
    "copy": ("Kus669/ViSceT5-0bbd575-vitextvqa-consformer-twc-p35-tau0.07-bs32x1", "checkpoint-5495"),
    "nocopy": ("Kus669/ViSceT5-b744ce2-vitextvqa-consformer-twc-p35-tau0.07-bs32x1", "checkpoint-5495"),
}
RELATED_MODES = ["correct", "augment", "none"]
CMP_BASE = ("Kus669/ViSceT5-cc773b0-vitextvqa-base-bs32x1", "predictions/checkpoint-5495/submission_test.csv")

import os, shutil, json
from huggingface_hub import hf_hub_download
!git checkout -q --detach {CODE_SHA}
TEST_GT_JSON = f"/content/ViSceT5/eval_gt/{DATASET}_test_gt.json"
cmp_tag = "" if DATASET == "ViTextVQA" else f"{DATASET}_"
cmp_preds = [f"base={hf_hub_download(*CMP_BASE)}"]
for cmp_name, (cmp_repo, cmp_ckpt) in HF_CMP_RUNS.items():
    cmp_dir = f"/content/ViSceT5/output/cmp_{cmp_repo.split('/')[-1]}"
    if os.path.exists("/content/cmp_ckpt_info.json"):
        os.remove("/content/cmp_ckpt_info.json")
    !python scripts/hf_io.py prepare --repo {cmp_repo} --dir {cmp_dir} --ckpt {cmp_ckpt} --info_out /content/cmp_ckpt_info.json
    CMP_INFO = json.load(open("/content/cmp_ckpt_info.json", encoding="utf-8"))
    cmp_model, cmp_repo_path = CMP_INFO["model_dir"], CMP_INFO["pred_path_in_repo"]
    assert json.load(open(os.path.join(cmp_model, "config.json"), encoding="utf-8")).get("use_twc"), "Checkpoint không có TWC"
    for mode in RELATED_MODES:
        mode_dir = os.path.join(CMP_INFO["pred_dir"], f"related_{mode}")
        os.makedirs(mode_dir, exist_ok=True)
        mode_pred = os.path.join(mode_dir, f"submission_{cmp_tag}test.csv")
        if not os.path.exists(mode_pred):
            !MOCK_TEST=false STAGE=predict PREDICT_SPLIT=both PREDICT_DATASET={DATASET} PREDICT_RELATED={mode} PREDICT_CKPT_DIR={cmp_model} HF_REPO= python3 run_pipeline.py 2>&1 | tee {mode_dir}/predict_log.txt
            for sp in ("dev", "test"):
                shutil.move(f"/content/ViSceT5/output/submission_{cmp_tag}{sp}.csv", os.path.join(mode_dir, f"submission_{cmp_tag}{sp}.csv"))
        print(cmp_name, mode, [l.strip() for l in open(f"{mode_dir}/predict_log.txt", encoding="utf-8", errors="ignore") if "[dev] EM=" in l][-1:])
        if os.environ.get("HF_TOKEN"):
            !python scripts/hf_io.py upload_predictions --repo {cmp_repo} --dir {mode_dir} --path_in_repo {cmp_repo_path}/related_{mode} --files submission_{cmp_tag}dev.csv submission_{cmp_tag}test.csv predict_log.txt
        cmp_preds.append(f"{cmp_name}_{mode}={mode_pred}")
CMP_ARG = " ".join(cmp_preds)
!python scripts/error_anatomy.py --gt {TEST_GT_JSON} --dataset {DATASET} --pred {CMP_ARG}


## 6d. Phân tích lỗi: chỉ dùng chữ, không cần GPU
Xếp mỗi câu test vào một nhóm theo đáp án đúng, các từ OCR mà model thấy và câu trả lời của model. Mỗi nhóm cho biết nên sửa phần nào:

| Nhóm | Ý nghĩa | Hướng sửa |
|---|---|---|
| `full_outside` | OCR có đủ đáp án, model sinh từ không có trong OCR | copy từ OCR |
| `full_boundary` | Từ đều từ OCR, trùng một phần đáp án | chọn đúng cụm |
| `full_region` | Từ đều từ OCR, không trùng đáp án | liên kết câu hỏi với OCR |
| `corrected_*` | Chỉ bản sửa lỗi mới đủ đáp án | giá trị của bản related |
| `partial_*`, `none_*` | OCR thiếu từ của đáp án | chất lượng OCR |

Cell tự lấy baseline `cc773b0` và các run Consformer, TWC, Full của commit ở cell 2 đã có file predict trên HF, với cùng `BATCH_SIZE`, `GRAD_ACCUM`, `TWC_AUG_P`, `TWC_TAU`. Run Full `1e5b8ef` luôn được thêm với tên `full_1e5b8ef`; khi `TWC_CONTRASTIVE=False`, run hiện tại có tên `no_twc`. Cuối bảng là chênh lệch EM có cặp giữa mọi cặp run.


In [ ]:
from huggingface_hub import HfApi, hf_hub_download
!git checkout -q --detach {CODE_SHA}
ANATOMY_RUNS = {"base": ("Kus669/ViSceT5-cc773b0-vitextvqa-base-bs32x1", "predictions/checkpoint-5495/submission_test.csv")}
_suffix = f"bs{BATCH_SIZE}x{GRAD_ACCUM}"
_twc = f"twc-p{int(round(100 * TWC_AUG_P))}-tau{TWC_TAU:g}"
for _name, _run in (("consformer", f"consformer-{_suffix}"), ("twc", f"{_twc}-{_suffix}"), ("full", f"consformer-{_twc}-{_suffix}")):
    _repo = f"{HF_USER}/ViSceT5-{CODE_TAG}-{DATASET.lower()}-{_run}"
    if HfApi().file_exists(_repo, "predictions/submission_test.csv"):
        ANATOMY_RUNS[_name] = (_repo, "predictions/submission_test.csv")
    else:
        print("Chưa có file predict:", _repo)
ANATOMY_RUNS["full_1e5b8ef"] = (f"{HF_USER}/ViSceT5-1e5b8ef-{DATASET.lower()}-consformer-twc-p35-tau0.07-bs32x1", "predictions/submission_test.csv")
if USE_TWC and not TWC_CONTRASTIVE and HfApi().file_exists(HF_REPO, "predictions/submission_test.csv"):
    ANATOMY_RUNS["no_twc"] = (HF_REPO, "predictions/submission_test.csv")
anatomy_preds = [f"{name}={hf_hub_download(repo, path)}" for name, (repo, path) in ANATOMY_RUNS.items()]
ANATOMY_ARG = " ".join(anatomy_preds)
!python scripts/error_anatomy.py --gt /content/ViSceT5/eval_gt/{DATASET}_test_gt.json --dataset {DATASET} --pred {ANATOMY_ARG}


## 7. Tổng hợp kết quả mọi cấu hình đã chạy

In [ ]:
import json, os, glob
import pandas as pd
rows = []
for od in sorted(glob.glob(f"/content/ViSceT5/output/*_{DATASET}_*")):
    if not os.path.isdir(od) or od.endswith("_mock"):
        continue
    tag, name = os.path.basename(od).split(f"_{DATASET}_", 1)
    mods = set(name.split("-"))
    r = {"code": tag, "run": name, "QA-ViT": "question" if "qavit" in mods else ("no_question" if "qavitnq" in mods else "off"),
         "Consformer": "consformer" in mods or "scespavis" in mods, "TWC": "twc" in mods}
    st = sorted(glob.glob(os.path.join(od, "checkpoint-*", "trainer_state.json")),
                key=lambda p: int(p.split("checkpoint-")[-1].split(os.sep)[0]))
    if st:
        hist = json.load(open(st[-1])).get("log_history", [])
        key = "eval_cider" if DATASET == "OpenViVQA" else "eval_em_f1"
        evs = [h for h in hist if "eval_em" in h]
        if evs:
            best = max(evs, key=lambda h: h.get(key, 0.5 * (h.get("eval_em", 0) + h.get("eval_f1", 0))))
            r.update({"val_EM": round(100 * best.get("eval_em", float("nan")), 2), "val_F1": round(100 * best.get("eval_f1", 0), 2),
                      "val_CIDEr": round(best.get("eval_cider", float("nan")), 4), "best_epoch": best.get("epoch")})
    for pd_dir in [od] + sorted(glob.glob(os.path.join(od, "predict_checkpoint-*"))):
        rr = dict(r, ckpt="best" if pd_dir == od else os.path.basename(pd_dir)[len("predict_"):])
        ev = os.path.join(pd_dir, "eval_test.json")
        if os.path.exists(ev):
            t = json.load(open(ev, encoding="utf-8"))
            rr.update({"test_EM": round(100 * t.get("Exact Match (EM)", float("nan")), 2), "test_F1": round(100 * t.get("F1 (Token-level)", float("nan")), 2),
                       "test_BLEU4": round(t.get("BLEU-4", float("nan")), 4), "test_CIDEr": round(t.get("CIDEr", float("nan")), 4),
                       "test_BERTScore": round(100 * t.get("BERTScore_F1", float("nan")), 2)})
        elif pd_dir != od:
            continue
        rows.append(rr)
df = pd.DataFrame(rows)
print(df.to_string(index=False) if len(df) else "Chưa có cấu hình nào chạy xong.")

## 8. Trực quan hoá attention (cấu hình hiện tại)
`scripts/visualize_attention.py` sinh đáp án (beam search như predict) rồi lấy **cross-attention** của các token đáp án lên đầu vào encoder,
trung bình trên mọi lớp và mọi head. Mỗi câu hỏi ra một hình: heatmap trên 14×14 patch ảnh (nếu mô hình dùng ảnh), box OCR tô màu theo attention,
và attention trên từng từ của câu hỏi, OCR gốc, OCR related (viền đỏ = từ bị đổi). `segment_mass.csv` ghi tỷ lệ attention dồn vào từng phần.
Dùng checkpoint `PREDICT_CKPT` (cell 2). Đặt `ATTN_IDS` để chọn câu hỏi cụ thể; để trống thì lấy ngẫu nhiên `ATTN_N` câu (cùng seed cho mọi cấu hình). Hình được upload vào `attention/` (hoặc `attention/checkpoint-N/`) trên repo HF.

In [ ]:
import os, glob
from IPython.display import Image, display
!git checkout -q --detach {CODE_SHA}
ATTN_SPLIT = "test"
ATTN_N = 8
ATTN_IDS = []

import json
if os.path.exists("/content/ckpt_info.json"):
    os.remove("/content/ckpt_info.json")
!python scripts/hf_io.py prepare --repo {HF_REPO} --dir {OUTPUT_DIR} --ckpt {PREDICT_CKPT} --info_out /content/ckpt_info.json
CKPT_INFO = json.load(open("/content/ckpt_info.json", encoding="utf-8"))
MODEL_DIR, ATTN_DIR = CKPT_INFO["model_dir"], os.path.join(CKPT_INFO["pred_dir"], "attention")
ATTN_REPO_PATH = CKPT_INFO["attn_path_in_repo"]
ids_arg = ("--ids " + " ".join(map(str, ATTN_IDS))) if ATTN_IDS else f"--n {ATTN_N}"
gt_arg = f"--gt eval_gt/{DATASET}_test_gt.json" if os.path.exists(f"eval_gt/{DATASET}_test_gt.json") else ""
!python scripts/visualize_attention.py --ckpt_dir {MODEL_DIR} --dataset {DATASET} --split {ATTN_SPLIT} {ids_arg} {gt_arg} --out_dir {ATTN_DIR}

pngs = sorted(glob.glob(os.path.join(ATTN_DIR, "*.png")))
for p in pngs:
    display(Image(filename=p))
if os.environ.get("HF_TOKEN") and HF_REPO and pngs:
    names = " ".join(os.path.basename(p) for p in pngs + [os.path.join(ATTN_DIR, "segment_mass.csv")])
    !python scripts/hf_io.py upload_predictions --repo {HF_REPO} --dir {ATTN_DIR} --path_in_repo {ATTN_REPO_PATH} --files {names}